In [ ]:
import pypsa
import matplotlib.pyplot as plt
import yaml

In [ ]:
n = pypsa.Network(
    "../../results/cost_year~2030/transport_cost~steel_r_iron_r/demand~1/product~steel/network.nc"
)

#### Shipping cost

In [ ]:
n.links[n.links.carrier == "shipping_steel"].marginal_cost / 2000

In [ ]:
n.links[n.links.carrier == "shipping_iron_ore"].marginal_cost

In [ ]:
n.objective / 1e9

In [ ]:
n.statistics()

In [ ]:
n.statistics()

In [ ]:
n.statistics.energy_balance()

### Global supply, demand and trade

#### Supply

In [ ]:
supply = n.statistics.energy_balance(comps=["Generator"], groupby=["bus"]).div(1e6)
supply.sort_values(ascending=False)

In [ ]:
print(f"The total supply is {supply.sum():.2f} Mt")

In [ ]:
# Most expensive supply of steel
n.links[n.links.carrier == "steel"].marginal_cost.sort_values() * 1.59 + (
    98 * 1.59
)  # .describe()*1.59

#### Demand

In [ ]:
demand = n.statistics.energy_balance(comps=["Load"], groupby=["bus"]).div(-1e6)
demand.sort_values(ascending=False)

In [ ]:
print(f"The total demand is {demand.sum():.2f} Mt")

#### Trade

In [ ]:
trade = n.statistics.energy_balance(comps=["Link"], groupby=["bus", "carrier"]).div(1e6)
trade

In [ ]:
trade.sum()

Exporters

In [ ]:
exporters = trade[trade < 0] * (-1)
exporters.sort_values(ascending=False)

Importers

In [ ]:
importers = trade[trade > 0]
importers.sort_values(ascending=False)

### Iron ore

In [ ]:
n.statistics.energy_balance(comps=["Link"], groupby=["bus", "carrier"]).div(1e6).loc[
    :, :, "shipping_iron_ore"
]

In [ ]:
n.statistics.energy_balance(comps=["Generator"], groupby=["bus", "carrier"]).div(1e6)

### Cost

In [ ]:
print(f"The total cost is {n.objective / 1e9:.2f} B EUR")

In [ ]:
n.statistics()

In [ ]:
n.generators.marginal_cost

In [ ]:
n.links.p_nom_opt.sort_values(ascending=False)

### Sankey

### Global map

In [ ]:
# Load config from YAML
with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Get plot config for steel
plot_config = config["plot"]["world_map"]["steel"]
colors = config["plot"]["colors"]

In [ ]:
product = "steel"

In [ ]:
import cartopy.crs as ccrs
import yaml


def plot_trade_map(
    n, supply, demand, trade, product="steel", alpha_supply=0.7, alpha_demand=1
):

    with open("../../config/config.yaml", "r") as f:
        config = yaml.safe_load(f)

    plot_config = config["plot"]["world_map"][product]
    colors = config["plot"]["colors"]

    supply_color = colors.get(f"{product}_supply", "black")
    demand_color = colors.get(f"{product}_demand", "lightsteelblue")
    link_colors = colors.get(f"{product}_link", "gray")

    fig = plt.figure(figsize=(10, 5))
    ax = plt.axes(projection=ccrs.PlateCarree())

    # ax.coastlines()

    # Plot demand
    n.plot.map(
        ax=ax,
        bus_sizes=demand * plot_config["bus_size"],
        bus_colors=demand_color,
        bus_alpha=alpha_demand,
        link_widths=0,
        branch_components=["Link"],
    )

    # Plot supply + trade
    n.plot.map(
        ax=ax,
        bus_sizes=supply * plot_config["bus_size"],
        bus_colors=supply_color,
        bus_alpha=alpha_supply,
        link_widths=trade * plot_config["link_width"],
        branch_components=["Link"],
        link_colors=link_colors,
    )

    ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
    # ax.set_global()

    # Legend
    legend_elements = [
        plt.Line2D([0], [0], color=link_colors, label="shipping"),
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="white",
            label="Demand",
            markerfacecolor=demand_color,
            markersize=10,
        ),
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="white",
            label="Supply",
            markerfacecolor=supply_color,
            markersize=10,
        ),
    ]

    fig.legend(
        handles=legend_elements,
        frameon=False,
        loc="lower right",
        bbox_to_anchor=(0.22, 0.28),
    )

    return fig

In [ ]:
# Supply and demand
steel_gen = (
    n.statistics.supply(comps=["Link"], groupby=["bus", "carrier"])
    .loc[:, :, "steel"]
    .droplevel(0)
)
steel_load = n.loads.groupby("bus").p_set.sum()
steel_trade = n.links[n.links.carrier == "shipping_steel"].p_nom_opt.astype(int)

# Plot
fig = plot_trade_map(
    n,
    supply=steel_gen,
    demand=steel_load,
    trade=steel_trade,
    product="steel",
    alpha_supply=0.7,
)

In [ ]:
# Supply and demand
iron_ore_gen = (
    n.statistics.supply(comps=["Generator"], groupby=["bus", "carrier"])
    .loc[:, :, "iron_ore"]
    .droplevel(0)
)
iron_ore_load = (
    n.statistics.withdrawal(comps=["Link"], groupby=["bus", "carrier"])
    .loc[:, :, "steel"]
    .droplevel(0)
)
iron_ore_trade = n.links[n.links.carrier == "shipping_iron_ore"].p_nom_opt.astype(int)

# Shipping color from carrier
color_shipping_iron_ore = n.carriers.loc["shipping_iron_ore", "color"]

# Plot
fig = plot_trade_map(
    n,
    supply=iron_ore_gen,
    demand=iron_ore_load,
    trade=iron_ore_trade,
    product="iron_ore",
    alpha_supply=0.5,
)